# cuda-empty-cache — faded example 1: Fill in the release-cadence condition

> Practice drill from [Delta Drills](https://delta-drills.vercel.app). Atom: `cuda-empty-cache`. The last cell reports your progress on the `PyTorch: torch.cuda.empty_cache` subtopic back to Delta Drills.

**Most of the code is already written — complete the one blanked step**, run the test to check it, then run the last cell to record your progress.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `PyTorch: torch.cuda.empty_cache` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`cuda-empty-cache`** (exercise 1). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "cuda-empty-cache"
DD_SUBTOPIC = "PyTorch: torch.cuda.empty_cache"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## Concept

_First time on this topic? Run the **Setup** cell above and skim it: every class and helper mentioned below is defined there. You don't need to have done any other drill first._

Releasing the CUDA cache every `K` iterations keeps reserved memory bounded in a long loop. The cadence is expressed as a modulo test on a 1-indexed counter so the release fires on multiples of `K`. The release never affects the per-iteration results, which are live tensors.

## Faded exercise 1

Complete `cadence_release(items, K)`. It computes each item's `.sum()` into a list and calls `t.cuda.empty_cache()` after every `K`-th item (1-indexed). You must supply the boolean condition that decides when to release.

**Your task:** complete the one blanked step in the code cell below. The surrounding code, function signatures, and variable names are given — work out the missing expression yourself, then run the test.

In [ ]:
def cadence_release(items, K: int):
    sums = []
    for i, x in enumerate(items, start=1):
        sums.append(x.sum())
        should_release = None  # TODO: fill in this step — read the prompt cell above
        if should_release:
            t.cuda.empty_cache()
    return t.stack(sums)

t.manual_seed(0)
items = [t.randn(6) for _ in range(5)]
result = cadence_release(items, K=2)


def _test():
    calls = {'n': 0}
    orig = t.cuda.empty_cache
    t.cuda.empty_cache = lambda: calls.__setitem__('n', calls['n'] + 1)
    try:
        t.manual_seed(0)
        items = [t.randn(6) for _ in range(5)]
        expected = t.stack([x.sum() for x in items])
        out = cadence_release(items, K=2)
        assert out.shape == (5,), out.shape
        assert t.allclose(out, expected), (out, expected)
        # blank load-bearing: release must fire exactly on multiples of K (i=2,4 -> 2 times)
        assert calls['n'] == 2, ('release count', calls['n'])
    finally:
        t.cuda.empty_cache = orig


try:
    _test()
    _dd_passed.add('faded1')
    print('[Delta Drills] faded1 passed.')
except AssertionError as _e:
    print('Test failed:', _e)

## Report your progress

Run the cell below to send your progress to Delta Drills. It only counts if the test above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'faded1'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:faded1',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()

<details><summary>Solution</summary>

```python
def cadence_release(items, K: int):
    sums = []
    for i, x in enumerate(items, start=1):
        sums.append(x.sum())
        should_release = (i % K == 0)
        if should_release:
            t.cuda.empty_cache()
    return t.stack(sums)

t.manual_seed(0)
items = [t.randn(6) for _ in range(5)]
result = cadence_release(items, K=2)
```
</details>